# TD3 baseline on Walker Walk

In [1]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from utils import TrajectoryReplayBuffer, collect_episode, evaluate_policy, set_seed
from visualisations import plot_policy_rollouts, plot_q_diagnostics
from benchmarks import make_env, load_dataset_into_buffer
from benchmarks.visualise import collect_episode_frames, save_gif
from networks import Factorised_TD3_Critic
from agents import Factorised_TD3_Actor
from trainer import td3_train

DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


Using device: mps


In [2]:
env = make_env(
    benchmark="exorl",
    task="walker_walk",
    seed=0,
    render_mode="rgb_array",
    render_height=240,
    render_width=320,
    camera_id=0,
)

rollout = collect_episode_frames(env, max_steps=200)
print(rollout["num_steps"], rollout["total_reward"])

save_gif(rollout["frames"], "outputs/walker_walk.gif", fps=30)
env.close()


print(env.observation_space)
print(env.action_space)

obs, info = env.reset(seed=42)
print(obs.shape, obs.dtype)

a = env.action_space.sample()
next_obs, reward, terminated, truncated, info = env.step(a)
print(next_obs.shape, reward, terminated, truncated)

200 7.318209249243061
Box(-inf, inf, (24,), float32)
Box(-1.0, 1.0, (6,), float32)
(24,) float32
(24,) 0.1611122317340134 False False


In [ ]:
# --- TD3 setup to match the passed td3_train(...) signature ---

seed = 42
benchmark = "exorl"
task = "walker_walk"
BUFFER_CAPACITY = 100_000

# Main training env
env = make_env(
    benchmark=benchmark,
    task=task,
    seed=seed,
)

obs_dim = env.observation_space.shape[0]
act_dim = env.action_space.shape[0]

print("obs_dim:", obs_dim, "act_dim:", act_dim)


goal = np.zeros(2, dtype=np.float32)

# Instantiate the networks expected by your trainer
actor_curr = Factorised_TD3_Actor(obs_dim, act_dim).to(DEVICE)
actor_tgt_curr = Factorised_TD3_Actor(obs_dim, act_dim).to(DEVICE)

q1_curr = Factorised_TD3_Critic(obs_dim, act_dim).to(DEVICE)
q2_curr = Factorised_TD3_Critic(obs_dim, act_dim).to(DEVICE)
q1_tgt_curr = Factorised_TD3_Critic(obs_dim, act_dim).to(DEVICE)
q2_tgt_curr = Factorised_TD3_Critic(obs_dim, act_dim).to(DEVICE)

# Your trainer expects make_env(goal=goal), but Walker Walk notebook usage is
# make_env(benchmark=..., task=..., seed=...). This wrapper adapts the signature.
def walker_make_env(goal=None):
    return make_env(
        benchmark=benchmark,
        task=task,
        seed=seed,
    )

(
    actor_trained,
    actor_tgt_trained,
    q1_main,
    q1_tgt_main,
    q2_trained,
    q2_tgt_trained,
    eval_returns,
    min_steps,
    min_time,
    task_embedding,
    sa_embedding_mean,
    sa_embedding_fixed,
    sa_batch_final,
    replay,
) = td3_train(
    seed=seed,
    actor=actor_curr,
    actor_tgt=actor_tgt_curr,
    q1=q1_curr,
    q2=q2_curr,
    q1_tgt=q1_tgt_curr,
    q2_tgt=q2_tgt_curr,
    env=env,
    make_env=walker_make_env,
    goal=goal,
    buffer_capacity=BUFFER_CAPACITY,
    total_steps=500_000,
    warmup_steps=10_000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    policy_noise=0.2,
    noise_clip=0.5,
    policy_delay=2,
    expl_noise=0.15,
    lr=3e-4,
    train_freq=1,
    gradient_steps=1,
    eval_every=5_000,
    device=DEVICE,
    obs_dim=obs_dim,
    act_dim=act_dim,
)
if eval_returns:
    xs, ys = zip(*eval_returns)
    plt.figure(figsize=(7, 4))
    plt.plot(xs, ys)
    plt.xlabel("Environment steps")
    plt.ylabel("Mean episodic return")
    plt.title("TD3 baseline on Walker Walk")
    plt.grid(alpha=0.25)
    plt.show()

obs_dim: 24 act_dim: 6


RuntimeError: linear(): input and weight.T shapes cannot be multiplied (1x25 and 26x256)

In [ ]:
eval_vis_env = make_env(
    benchmark="exorl",
    task="walker_walk",
    seed=123,
    render_mode="rgb_array",
    render_height=240,
    render_width=320,
    camera_id=0,
)

rollout = collect_episode_frames(
    eval_vis_env,
    policy_fn=lambda obs: (
        td3_actor(
            torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        ).squeeze(0).detach().cpu().numpy().astype(np.float32)
    ),
    max_steps=500,
)

print(rollout["num_steps"], rollout["total_reward"])
save_gif(rollout["frames"], "outputs/walker_walk_td3_trained.gif", fps=30)
eval_vis_env.close()